# MiniMax-M2.7 Cookbook

A practical notebook for running MiniMax-M2.7 with Transformers and OpenAI-compatible tool calling flows.

Source links:
- Hugging Face model card: https://huggingface.co/MiniMaxAI/MiniMax-M2.7
- Transformers deploy guide: https://huggingface.co/MiniMaxAI/MiniMax-M2.7/blob/main/docs/transformers_deploy_guide.md
- Tool calling guide: https://huggingface.co/MiniMaxAI/MiniMax-M2.7/blob/main/docs/tool_calling_guide.md

## 0) Setup and Installation

MiniMax reports these deployment requirements for local Transformers usage:
- OS: Linux
- Python: 3.9 to 3.12
- Transformers: 4.57.1
- GPU memory for weights: about 220 GB

If your environment is smaller, use remote inference instead of local loading.

In [ ]:
MODEL = "MiniMaxAI/MiniMax-M2.7"
DEFAULT_SYSTEM_PROMPT = "You are a helpful assistant. Your name is MiniMax-M2.7 and is built by MiniMax."
RECOMMENDED_GENERATION = {
    "temperature": 1.0,
    "top_p": 0.95,
    "top_k": 40,
}

print(MODEL)
print(RECOMMENDED_GENERATION)

In [ ]:
# Uncomment and run in a fresh environment when needed.
# !uv pip install transformers==4.57.1 torch accelerate --torch-backend=auto

# If you have Hugging Face network issues, set mirror endpoint before loading:
# import os
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

## 1) Basic Usage with Provider SDK (Transformers)

This follows the official Transformers deployment guide for MiniMax-M2.7.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    device_map="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL)

print(type(model).__name__)
print(type(tokenizer).__name__)

In [ ]:
messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": DEFAULT_SYSTEM_PROMPT}],
    },
    {
        "role": "user",
        "content": [{"type": "text", "text": "Give me a 3-step plan to debug a flaky CI pipeline."}],
    },
]

model_inputs = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt",
    add_generation_prompt=True,
)

# Keep the input on GPU if available.
if torch.cuda.is_available():
    model_inputs = model_inputs.to("cuda")

generated_ids = model.generate(
    model_inputs,
    max_new_tokens=256,
    generation_config=model.generation_config,
)

response = tokenizer.batch_decode(generated_ids)[0]
print(response[:2000])

## 2) Framework Integration

MiniMax provides tool-calling examples via OpenAI-compatible chat completions.
Use this when serving the model through an OpenAI-compatible endpoint such as local vLLM/SGLang.

In [ ]:
# Uncomment to install if needed:
# !pip install openai

import json
from typing import Any, cast
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="dummy")

tools: list[dict[str, Any]] = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the current weather in a given location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "City and state, e.g., San Francisco, CA"},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
            },
            "required": ["location", "unit"],
        },
    },
}]

response = client.chat.completions.create(
    model=client.models.list().data[0].id,
    messages=[{"role": "user", "content": "What is the weather in San Francisco? Use celsius."}],
    tools=cast(Any, tools),
    tool_choice="auto",
)

tool_calls = response.choices[0].message.tool_calls or []
if not tool_calls:
    print("No tool call returned")
else:
    function_obj = getattr(tool_calls[0], "function", None)
    if function_obj is None:
        print("First tool call does not contain a function payload")
    else:
        print("Function called:", function_obj.name)
        print("Arguments:", function_obj.arguments)
        print("Parsed args:", json.loads(function_obj.arguments))

## 3) Building Agents

MiniMax-M2.7 can emit structured XML tool calls. If your serving stack does not parse these natively, parse them manually as documented.

In [ ]:
import json
import re
from typing import Any, Dict, List, Optional

def extract_name(name_str: str) -> str:
    name_str = name_str.strip()
    if name_str.startswith('\"') and name_str.endswith('\"'):
        return name_str[1:-1]
    if name_str.startswith("'") and name_str.endswith("'"):
        return name_str[1:-1]
    return name_str

def convert_param_value(value: str, param_type: str) -> Any:
    if value.lower() == "null":
        return None

    p = param_type.lower()
    if p in ["string", "str", "text"]:
        return value
    if p in ["integer", "int"]:
        try:
            return int(value)
        except (TypeError, ValueError):
            return value
    if p in ["number", "float"]:
        try:
            val = float(value)
            return val if val != int(val) else int(val)
        except (TypeError, ValueError):
            return value
    if p in ["boolean", "bool"]:
        return value.lower() in ["true", "1"]

    try:
        return json.loads(value)
    except json.JSONDecodeError:
        return value

def parse_tool_calls(model_output: str, tools: Optional[List[Dict]] = None) -> List[Dict]:
    if "<minimax:tool_call>" not in model_output:
        return []

    tool_calls: List[Dict] = []
    tool_call_regex = re.compile(r"<minimax:tool_call>(.*?)</minimax:tool_call>", re.DOTALL)
    invoke_regex = re.compile(r"<invoke name=(.*?)</invoke>", re.DOTALL)
    parameter_regex = re.compile(r"<parameter name=(.*?)</parameter>", re.DOTALL)

    for tool_block in tool_call_regex.findall(model_output):
        for invoke_block in invoke_regex.findall(tool_block):
            name_match = re.search(r"^([^>]+)", invoke_block)
            if not name_match:
                continue

            function_name = extract_name(name_match.group(1))
            param_config = {}

            if tools:
                for tool in tools:
                    t_name = tool.get("name") or tool.get("function", {}).get("name")
                    if t_name == function_name:
                        params = tool.get("parameters") or tool.get("function", {}).get("parameters")
                        if isinstance(params, dict):
                            param_config = params.get("properties", {})
                        break

            args: Dict[str, Any] = {}
            for match in parameter_regex.findall(invoke_block):
                param_match = re.search(r"^([^>]+)>(.*)", match, re.DOTALL)
                if not param_match:
                    continue

                param_name = extract_name(param_match.group(1))
                raw_value = param_match.group(2).strip()
                param_type = param_config.get(param_name, {}).get("type", "string") if isinstance(param_config.get(param_name), dict) else "string"
                args[param_name] = convert_param_value(raw_value, param_type)

            tool_calls.append({"name": function_name, "arguments": args})

    return tool_calls

In [ ]:
sample_tools = [{
    "name": "get_weather",
    "parameters": {
        "type": "object",
        "properties": {
            "location": {"type": "string"},
            "unit": {"type": "string"},
        },
        "required": ["location", "unit"],
    },
}]

sample_output = """<minimax:tool_call>
<invoke name=\"get_weather\">
<parameter name=\"location\">San Francisco</parameter>
<parameter name=\"unit\">celsius</parameter>
</invoke>
</minimax:tool_call>"""

parsed = parse_tool_calls(sample_output, sample_tools)
print(parsed)

## 4) Advanced Applications

This section demonstrates practical patterns for evaluation, batching, and safer generation settings.

In [ ]:
def build_messages(task: str) -> list:
    return [
        {
            "role": "system",
            "content": [{"type": "text", "text": DEFAULT_SYSTEM_PROMPT}],
        },
        {
            "role": "user",
            "content": [{"type": "text", "text": task}],
        },
    ]

tasks = [
    "Summarize this incident report into 5 action items.",
    "Write a minimal rollback checklist for a failed deploy.",
    "List the top 3 failure modes in this release process.",
]

print(f"Prepared {len(tasks)} prompts")

In [ ]:
results = []
for task in tasks:
    encoded = tokenizer.apply_chat_template(
        build_messages(task),
        return_tensors="pt",
        add_generation_prompt=True,
    )

    if torch.cuda.is_available():
        encoded = encoded.to("cuda")

    output_ids = model.generate(
        encoded,
        max_new_tokens=180,
        generation_config=model.generation_config,
    )
    text = tokenizer.batch_decode(output_ids)[0]
    results.append({"task": task, "response": text[:800]})

for i, item in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print("Task:", item["task"])
    print("Response preview:", item["response"])

In [ ]:
import asyncio
from openai import AsyncOpenAI

aclient = AsyncOpenAI(base_url="http://localhost:8000/v1", api_key="dummy")

async def run_async_prompt(prompt: str):
    return await aclient.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=RECOMMENDED_GENERATION["temperature"],
        top_p=RECOMMENDED_GENERATION["top_p"],
    )

async def main():
    prompts = [
        "Give me a concise on-call handoff template.",
        "Write a release readiness checklist for backend services.",
    ]
    responses = await asyncio.gather(*[run_async_prompt(p) for p in prompts])

    for p, r in zip(prompts, responses):
        print("\nPrompt:", p)
        print("Completion:", r.choices[0].message.content)

# await main()

## Notes and Limits

- Local full-weight inference is resource intensive (reported around 220 GB for weights).
- If your stack supports built-in MiniMax tool-call parsing (for example vLLM or SGLang per vendor docs), prefer that over manual parsing.
- Keep `trust_remote_code=True` when loading this model with Transformers to avoid unsupported-model errors.
- Suggested baseline decoding settings from the model page are `temperature=1.0`, `top_p=0.95`, and `top_k=40`.

## Footer

You can now adapt these cells into your own evaluation harness, production prompts, and tool execution loop.